In [ ]:
!pip install adapters -q

In [2]:
import pandas as pd
from transformers import TrainingArguments, EarlyStoppingCallback, AutoTokenizer, set_seed
from datasets import Dataset
import joblib
import numpy as np
from google.colab import drive
import os
import json
import zipfile
from sklearn.metrics import f1_score, classification_report
import time
import torch
from transformers.trainer_utils import get_last_checkpoint
import adapters
from adapters import AutoAdapterModel,  DoubleSeqBnConfig, AdapterTrainer
#DoubleSeqBnConfig = Houlsby

In [3]:
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Houlsby"
os.makedirs(output_dir, exist_ok=True)

Mounted at /content/drive


**Dataset loading**

In [4]:
with zipfile.ZipFile("aapd.zip") as z:
    with z.open("aapd.json") as f:
        aapd = json.load(f)

In [5]:
aapd_df_train = pd.DataFrame(aapd["data"]["train"])
aapd_df_val = pd.DataFrame(aapd["data"]["val"])
aapd_df_test = pd.DataFrame(aapd["data"]["test"])

In [6]:
mlb = joblib.load("mlb.joblib")

In [7]:
#reusing the aapd's mlb
aapd_y_train = mlb.transform(aapd_df_train["labels"])
aapd_y_val   = mlb.transform(aapd_df_val["labels"])
aapd_y_test  = mlb.transform(aapd_df_test["labels"])

In [8]:
aapd_y_train.shape, aapd_y_val.shape, aapd_y_test.shape #ok

((53840, 54), (1000, 54), (1000, 54))

In [9]:
aapd_X_train = aapd_df_train["text"]
aapd_X_val   = aapd_df_val["text"]
aapd_X_test  = aapd_df_test["text"]

In [10]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
# for DistilBER max token length is 512 - the longest abstract has 522 words, so truncation will happen
def tokenize(texts):
    return tokenizer(texts.tolist(), padding="max_length", truncation=True, max_length=512)

train_enc = tokenize(aapd_X_train)
dev_enc   = tokenize(aapd_X_val)
test_enc  = tokenize(aapd_X_test)

In [13]:
y_train_bin = aapd_y_train.astype(np.float32)
y_dev_bin   = aapd_y_val.astype(np.float32)
y_test_bin  = aapd_y_test.astype(np.float32)

print(mlb.classes_)
print(y_train_bin.shape)

['Adaptation and Self-Organizing Systems' 'Applications'
 'Artificial Intelligence' 'Combinatorics' 'Computation and Language'
 'Computational Complexity'
 'Computational Engineering, Finance, and Science'
 'Computational Geometry' 'Computational Linguistics'
 'Computer Science and Game Theory'
 'Computer Vision and Pattern Recognition' 'Computers and Society'
 'Cryptography and Security' 'Data Analysis, Statistics and Probability'
 'Data Structures and Algorithms' 'Databases' 'Digital Libraries'
 'Discrete Mathematics' 'Disordered Systems and Neural Networks'
 'Distributed, Parallel, and Cluster Computing'
 'Formal Languages and Automata Theory' 'Human-Computer Interaction'
 'Information Retrieval' 'Information Theory (Computer Science)'
 'Information Theory (Mathematics)' 'Logic' 'Logic in Computer Science'
 'Machine Learning (Computer Science)' 'Machine Learning (Statistics)'
 'Mathematical Software' 'Methodology' 'Multiagent Systems' 'Multimedia'
 'Networking and Internet Architect

In [14]:
id2label = {i: label for i, label in enumerate(mlb.classes_)}
label2id = {label: i for i, label in enumerate(mlb.classes_)}

In [15]:
train_dataset = Dataset.from_dict({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "labels": y_train_bin.astype("float32")})

eval_dataset = Dataset.from_dict({
    "input_ids": dev_enc["input_ids"],
    "attention_mask": dev_enc["attention_mask"],
    "labels": y_dev_bin.astype("float32")})

test_dataset = Dataset.from_dict({
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "labels": y_test_bin.astype("float32")})

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))   #sigmoid
    preds = (probs >= 0.5).astype(int) #default threshold

    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {"f1_micro": f1_micro, "f1_macro": f1_macro}

In [18]:
print("Train labels:", y_train_bin.shape)
print("Val labels:", y_dev_bin.shape)
print("Test labels:", y_test_bin.shape)
print("Number of labels:", len(mlb.classes_))

Train labels: (53840, 54)
Val labels: (1000, 54)
Test labels: (1000, 54)
Number of labels: 54


**Training function**

In [19]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def reset_cuda_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

#measure vram only for final best config
def get_peak_vram_gb():
    if not torch.cuda.is_available():
        return None

    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def run_training(config, seed=0, evaluate_test=False, measure_vram=False, save_report=False):
    set_seed(seed)

    if measure_vram:
        reset_cuda_peak_memory()

    model = AutoAdapterModel.from_pretrained(config["base_model"])
    #remove unnecessary default/pretraining head as a check revealed it is present
    if "default" in model.heads:
        model.delete_head("default")

    #the adapters' library developers wrote on github that multilabel classification head is supported out of the box and can be implemented in the following way
    model.add_classification_head("aapd", num_labels=len(mlb.classes_),  multilabel=True, id2label=id2label)

    adapter_config = DoubleSeqBnConfig(reduction_factor=config["reduction_factor"])
    model.add_adapter("aapd", config=adapter_config, set_active=True)
    model.train_adapter("aapd")

    #check whether the adapters are working
    print("Active adapters:", model.active_adapters)
    print(model.adapter_summary())
    #check the classification head
    print("Trainable classification/head parameters:")
    for name, param in model.named_parameters():
        if param.requires_grad and ("head" in name.lower() or "classifier" in name.lower() or "classification" in name.lower()):
           print(name, param.numel())

    #for the final statistics
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    training_args = TrainingArguments(
        output_dir=config["output_dir"],

        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],

        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,

        save_total_limit=2,
        fp16=True,
        report_to="none")
    #adapter trainer, as recommended in the library's documentation
    trainer = AdapterTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"])])

    #check
    print("Active adapters after trainer creation:", trainer.model.active_adapters)

    #for resuming if something goes wrong/collab's runtime gets disconnected
    last_checkpoint = None
    if os.path.isdir(config["output_dir"]):
        last_checkpoint = get_last_checkpoint(config["output_dir"])

    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
    else:
        print("Starting training from scratch.")


    sync_cuda()
    train_start = time.perf_counter()
    #includes training + epoch validation + checkpoint saving + early stopping + loading best model
    trainer.train(resume_from_checkpoint=last_checkpoint)

    sync_cuda()
    train_time_sec = time.perf_counter() - train_start
    if measure_vram:
        training_peak_vram_gb = get_peak_vram_gb()
    else:
        training_peak_vram_gb = None

    sync_cuda()
    val_start = time.perf_counter()

    val_results = trainer.evaluate(eval_dataset, metric_key_prefix="val")

    sync_cuda()
    val_eval_time_sec = time.perf_counter() - val_start

    result = {
        "model": config["base_model"],
        "dataset": "AAPD",
        "method": "houlsby",
        "seed": seed,

        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "num_train_epochs": config["num_train_epochs"],

        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "actual_epochs_trained": trainer.state.epoch,


        "train_time_sec": train_time_sec,
        "val_eval_time_sec": val_eval_time_sec,

        "training_peak_vram_gb": training_peak_vram_gb,

        "trainable_params": trainable_params,
        "total_params": total_params,

        "val_f1_macro": val_results["val_f1_macro"],
        "val_f1_micro": val_results["val_f1_micro"]}

    if evaluate_test:
        sync_cuda()
        test_start = time.perf_counter()

        #single forward pass,  gives metrics + raw predictions
        test_pred_output = trainer.predict(test_dataset)

        sync_cuda()
        test_eval_time_sec = time.perf_counter() - test_start

        #derive predictions, needed for classification report
        test_probs = 1 / (1 + np.exp(-test_pred_output.predictions))
        test_binary_preds = (test_probs >= 0.5).astype(int)
        gold_labels = (test_pred_output.label_ids >= 0.5).astype(int)

        test_metrics = test_pred_output.metrics

        result.update({
            "test_eval_time_sec": test_eval_time_sec,
            "test_inference_per_sample_ms": (test_eval_time_sec / len(test_dataset)) * 1000,
            "test_f1_macro": test_metrics["test_f1_macro"],
            "test_f1_micro": test_metrics["test_f1_micro"],
            "avg_predicted_labels": float(test_binary_preds.sum(axis=1).mean()),
            "avg_gold_labels": float(gold_labels.sum(axis=1).mean()),})

        if save_report:
            report_dict = classification_report(
                gold_labels, test_binary_preds,
                target_names=mlb.classes_, zero_division=0, output_dict=True)
            report_df = pd.DataFrame(report_dict).T
            report_path = os.path.join(output_dir, f"classification_report_seed_{seed}.csv")
            report_df.to_csv(report_path)
            print(f"Classification report saved to {report_path}")

            #saving raw arrays for possible future analysis
            predictions_path = os.path.join(output_dir, f"test_predictions_seed_{seed}.npz")
            np.savez_compressed(
                predictions_path,
                y_true=gold_labels,
                y_pred=test_binary_preds,
                y_prob=test_probs,
                label_names=np.array(mlb.classes_),
                threshold=np.array([0.5]))
            result["test_predictions_path"] = predictions_path
            print(f"Predictions saved to {predictions_path}")

            #for saving the adapter
            adapter_save_path = os.path.join(config["output_dir"], "final_Houlsby_adapter_with_head")
            trainer.model.save_adapter(adapter_save_path, "aapd", with_head=True)
            result["saved_adapter_path"] = adapter_save_path
            print(f"Adapter + head saved to {adapter_save_path}")


    result["total_measured_time_sec"] = (result["train_time_sec"] + result["val_eval_time_sec"] + result.get("test_eval_time_sec", 0))

    return result

**Hyperparameter search**

In [20]:
#fixed params
base_config = {
    "output_dir": os.path.join(output_dir, "search"),
    "base_model": "distilbert-base-uncased",
    "tokenizer_name": "distilbert-base-uncased",
    "max_length": 512,
    "num_train_epochs": 10, #should be enough for such a large dataset
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "early_stopping_patience": 3,
    #adapter specific param:
    "reduction_factor": 8}  #the default one is 16, but since DistilBERT is already a smaller model I will go with 8, this is also the choice of  Razuvayevskaya et al. (2024)

#small search on the most relevant hyperparameters
learning_rates = [1e-4, 2e-4, 5e-4] #1e-4 is recommmended in the adapters library documentation, 2e-4 is used by Razuvayevskaya et al. (2024)
batch_sizes = [8, 16]


search_results_path = os.path.join(output_dir, "search_results.csv")
search_results = []

for lr in learning_rates:
    for bs in batch_sizes:
        config = base_config.copy()
        config["learning_rate"] = lr
        config["batch_size"] = bs
        config["output_dir"] = (f"{base_config['output_dir']}/lr_{lr}_bs_{bs}")

        #skipping already-completed configs when resuming
        if os.path.exists(search_results_path):
            existing = pd.read_csv(search_results_path)
            already_done = existing[
                (existing["learning_rate"] == lr) &
                (existing["batch_size"] == bs)]
            if len(already_done) > 0:
                print(f"Skipping lr={lr}, bs={bs} (already done)")
                search_results.append(already_done.iloc[0].to_dict())
                continue

        print("=" * 80)
        print(f"Running DistilBERT: lr={lr}, batch_size={bs}")
        print("=" * 80)

        result = run_training(config, seed=0)
        search_results.append(result)

        #saving incrementally after every config
        pd.DataFrame(search_results).to_csv(search_results_path, index=False)

search_results_df = pd.DataFrame(search_results)
search_results_df = search_results_df.sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
search_results_df.to_csv(search_results_path, index=False)

#!!! train_time_sec in search results is unreliable due to checkpoint resumption (when collab's runtime gets disconnected) !!!
# !!!but timing is only reported from the final seed runs - I ensured the run is not resumed

#the warning about adapters can be ignored, based on the check the adapters work
#the warining about early stopping can also be ignored, it works
search_results_df

Skipping lr=0.0001, bs=8 (already done)
Skipping lr=0.0001, bs=16 (already done)
Skipping lr=0.0002, bs=8 (already done)
Skipping lr=0.0002, bs=16 (already done)
Running DistilBERT: lr=0.0005, batch_size=8


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Active adapters: Stack[aapd]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
aapd                     bottleneck        1,779,840       2.682       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.aapd.1.weight 589824
heads.aapd.1.bias 768
heads.aapd.4.weight 41472
heads.aapd.4.bias 54
Active adapters after trainer creation: Stack[aapd]
Resuming from checkpoint: /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Houlsby/search/lr_0.0005_bs_8/checkpoint-60570


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
10,0.035200,0.062483,0.757381,0.590725


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Running DistilBERT: lr=0.0005, batch_size=16


Active adapters: Stack[aapd]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
aapd                     bottleneck        1,779,840       2.682       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.aapd.1.weight 589824
heads.aapd.1.bias 768
heads.aapd.4.weight 41472
heads.aapd.4.bias 54
Active adapters after trainer creation: Stack[aapd]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.117600,0.076776,0.652363,0.373062
2,0.071000,0.065504,0.710495,0.499746
3,0.064600,0.061875,0.731541,0.532153
4,0.059500,0.060288,0.742040,0.541412
5,0.054800,0.060175,0.745801,0.575793
6,0.050300,0.059532,0.745809,0.559957
7,0.045900,0.060648,0.746679,0.588366
8,0.041700,0.062445,0.751678,0.589615
9,0.037800,0.063704,0.755892,0.600185
10,0.034700,0.064364,0.752208,0.594710


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,train_time_sec,val_eval_time_sec,training_peak_vram_gb,trainable_params,total_params,val_f1_macro,val_f1_micro,total_measured_time_sec
0,distilbert-base-uncased,AAPD,houlsby,0,0.0005,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.600185,10.0,5822.589535,4.891844,NaN,2411958,68774838,0.600185,0.755892,5827.481379
1,distilbert-base-uncased,AAPD,houlsby,0,0.0005,8,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.598402,10.0,695.112727,9.339982,NaN,2411958,68774838,0.598402,0.756817,704.452709
2,distilbert-base-uncased,AAPD,houlsby,0,0.0002,8,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.587488,10.0,6072.255970,5.069332,NaN,2411958,68774838,0.587488,0.757394,6077.325302
3,distilbert-base-uncased,AAPD,houlsby,0,0.0002,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.563700,8.0,4715.552700,4.831505,NaN,2411958,68774838,0.563700,0.745608,4720.384205
4,distilbert-base-uncased,AAPD,houlsby,0,0.0001,8,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.554707,10.0,6104.121177,5.070288,NaN,2411958,68774838,0.554707,0.745107,6109.191465
5,distilbert-base-uncased,AAPD,houlsby,0,0.0001,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.532014,10.0,5929.544107,4.901706,NaN,2411958,68774838,0.532014,0.735675,5934.445813


**Best configuration**

In [21]:
best_row = search_results_df.iloc[0]

best_lr = float(best_row["learning_rate"])
best_batch_size = int(best_row["batch_size"])

print("Best learning rate:", best_lr)
print("Best batch size:", best_batch_size)
print("Best validation macro-F1:", best_row["val_f1_macro"])
print("Best checkpoint:", best_row["best_checkpoint"])

Best learning rate: 0.0005
Best batch size: 16
Best validation macro-F1: 0.600185187027472
Best checkpoint: /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Houlsby/search/lr_0.0005_bs_16/checkpoint-30285


In [22]:
best_config = base_config.copy()
best_config["learning_rate"] = best_lr
best_config["batch_size"] = best_batch_size
best_config["selection_metric"] = "val_f1_macro"
best_config["best_validation_macro_f1"] = float(best_row["val_f1_macro"])
best_config["best_validation_micro_f1"] = float(best_row["val_f1_micro"])
best_config["best_checkpoint_from_search"] = best_row["best_checkpoint"]

best_config_path = os.path.join(output_dir, "best_config.json")

with open(best_config_path, "w") as f:
    json.dump(best_config, f, indent=2)


**Final run (test set) on the best found configuration**

In [23]:
with open(best_config_path, "r") as f:
    final_config = json.load(f)

In [24]:
test_output_dir = os.path.join(output_dir, "test")
os.makedirs(test_output_dir, exist_ok=True)

test_results_path = os.path.join(test_output_dir, "AAPD_DistilBERT_Houlsby_test_results.csv")
test_results = []

for seed in [0, 1, 2]:
    config = final_config.copy()
    config["seed"] = seed
    config["output_dir"] = (os.path.join(test_output_dir, f"AAPD_DistilBERT_Houlsby_test_seed_{seed}"))

    #skips already completed seeds on resume
    if os.path.exists(test_results_path):
        existing = pd.read_csv(test_results_path)
        already_done = existing[existing["seed"] == seed]
        if len(already_done) > 0:
            print(f"Skipping seed={seed} (already done)")
            test_results.append(already_done.iloc[0].to_dict())
            continue

    print("=" * 80)
    print(f"Final run: seed={seed}, lr={final_config['learning_rate']}, batch_size={final_config['batch_size']}")
    print("=" * 80)
    result = run_training(config, seed=seed, evaluate_test=True, measure_vram=True, save_report = True)
    test_results.append(result)

    #saves incrementally after every seed
    pd.DataFrame(test_results).to_csv(test_results_path, index=False)

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(test_results_path, index=False)
test_results_df

Final run: seed=0, lr=0.0005, batch_size=16


Active adapters: Stack[aapd]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
aapd                     bottleneck        1,779,840       2.682       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.aapd.1.weight 589824
heads.aapd.1.bias 768
heads.aapd.4.weight 41472
heads.aapd.4.bias 54
Active adapters after trainer creation: Stack[aapd]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.117600,0.076772,0.651891,0.372910
2,0.071000,0.065489,0.710798,0.499936
3,0.064600,0.061878,0.732587,0.534309
4,0.059500,0.060200,0.743852,0.540443
5,0.054800,0.060246,0.741415,0.563168
6,0.050300,0.059474,0.748413,0.562877
7,0.045900,0.060233,0.755506,0.586142
8,0.041700,0.062706,0.748330,0.584763
9,0.037800,0.063577,0.751000,0.592958
10,0.034700,0.064293,0.753413,0.594954


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Classification report saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Houlsby/classification_report_seed_0.csv
Predictions saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Houlsby/test_predictions_seed_0.npz
Adapter + head saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Houlsby/test/AAPD_DistilBERT_Houlsby_test_seed_0/final_Houlsby_adapter_with_head
Final run: seed=1, lr=0.0005, batch_size=16


...

Active adapters: Stack[aapd]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
aapd                     bottleneck        1,779,840       2.682       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.aapd.1.weight 589824
heads.aapd.1.bias 768
heads.aapd.4.weight 41472
heads.aapd.4.bias 54
Active adapters after trainer creation: Stack[aapd]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.118000,0.077766,0.643373,0.379836
2,0.071000,0.066131,0.708813,0.483047
3,0.064600,0.063434,0.728146,0.519555
4,0.059500,0.059821,0.741765,0.533809
5,0.054900,0.059571,0.747592,0.578284
6,0.050300,0.060076,0.750000,0.589061
7,0.046000,0.060593,0.746972,0.573299
8,0.041600,0.060819,0.753281,0.581755
9,0.038000,0.061790,0.755842,0.588791


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Classification report saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Houlsby/classification_report_seed_2.csv
Predictions saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Houlsby/test_predictions_seed_2.npz
Adapter + head saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Houlsby/test/AAPD_DistilBERT_Houlsby_test_seed_2/final_Houlsby_adapter_with_head


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,...,val_f1_micro,test_eval_time_sec,test_inference_per_sample_ms,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,test_predictions_path,saved_adapter_path,total_measured_time_sec
0,distilbert-base-uncased,AAPD,houlsby,0,0.0005,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.594954,10.0,...,0.753413,4.777983,4.777983,0.594447,0.738679,2.106,2.421,/content/drive/MyDrive/thesis_results/AAPD_Dis...,/content/drive/MyDrive/thesis_results/AAPD_Dis...,5803.338205
1,distilbert-base-uncased,AAPD,houlsby,1,0.0005,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.590087,10.0,...,0.750000,4.899973,4.899973,0.555402,0.722799,2.009,2.421,/content/drive/MyDrive/thesis_results/AAPD_Dis...,/content/drive/MyDrive/thesis_results/AAPD_Dis...,5811.831565
2,distilbert-base-uncased,AAPD,houlsby,2,0.0005,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.589061,9.0,...,0.750000,5.110162,5.110162,0.567220,0.731553,1.997,2.421,/content/drive/MyDrive/thesis_results/AAPD_Dis...,/content/drive/MyDrive/thesis_results/AAPD_Dis...,5217.637000


In [25]:
test_summary_df = test_results_df[[
    "test_f1_macro",
    "test_f1_micro",
    "avg_predicted_labels",
    "avg_gold_labels",
    "train_time_sec",
    "val_eval_time_sec",
    "test_eval_time_sec",
    "test_inference_per_sample_ms",
    "training_peak_vram_gb",
    "actual_epochs_trained",
    "trainable_params",
    "total_params",
    "total_measured_time_sec"]].agg(["mean", "std"])

test_summary_path =  os.path.join(test_output_dir, "AAPD_DistilBERT_Houlsby_test_results_summary.csv")
test_summary_df.to_csv(test_summary_path)

test_summary_df

,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,train_time_sec,val_eval_time_sec,test_eval_time_sec,test_inference_per_sample_ms,training_peak_vram_gb,actual_epochs_trained,trainable_params,total_params,total_measured_time_sec
mean,0.572356,0.731010,2.037333,2.421,5601.204323,4.801894,4.929373,4.929373,1.565009,9.666667,2411958.0,68774838.0,5610.935590
std,0.020023,0.007954,0.059769,0.000,340.768392,0.105055,0.168030,0.168030,0.000813,0.577350,0.0,0.0,340.633043
